# 05 — Agent Composition: Using an Agent as a Tool

**What you'll learn:**
- How to convert an agent into a callable tool with `.as_tool()`
- Building specialist sub-agents and an orchestrator
- When to use agent composition vs multi-agent workflows

---

## Why Compose Agents?

In insurance underwriting, no single prompt can cover everything:
- **Risk assessment** requires medical/actuarial expertise
- **Policy recommendation** requires product knowledge
- **The underwriter** orchestrates both and makes a final decision

Agent Framework lets you build **specialist agents** and wire them together
using `.as_tool()`. The orchestrator agent calls specialists just like
any other function tool.

In [1]:
import os

from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv

from agent_framework.azure import AzureOpenAIResponsesClient

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()

## Step 1: Create Specialist Agents

Each specialist is a standalone agent with focused instructions.

In [2]:
def make_client():
    return AzureOpenAIResponsesClient(
        project_endpoint=PROJECT_ENDPOINT,
        deployment_name=DEPLOYMENT_NAME,
        credential=credential,
    )


# Specialist 1: Risk Assessment
risk_agent = make_client().as_agent(
    name="RiskAssessmentAgent",
    description="Evaluates the risk profile of an insurance applicant based on health, lifestyle, and demographic factors.",
    instructions=(
        "You are an actuarial risk assessment specialist. Given an applicant's profile, "
        "evaluate their risk level for life insurance. Consider:\n"
        "- Age and gender\n"
        "- Smoking status and BMI\n"
        "- Family medical history\n"
        "- Occupation hazards\n\n"
        "Provide a risk rating (low / moderate / high / very high) with "
        "a brief explanation of key risk factors. Be concise — 3-4 sentences max."
    ),
)

print(f"Created: {risk_agent.name}")

Created: RiskAssessmentAgent


In [3]:
# Specialist 2: Policy Recommendation
policy_agent = make_client().as_agent(
    name="PolicyRecommendationAgent",
    description="Recommends suitable insurance policy types and coverage levels based on an applicant's needs and risk profile.",
    instructions=(
        "You are an insurance product specialist. Given an applicant's profile and "
        "needs, recommend:\n"
        "1. Policy type (term life, whole life, universal life)\n"
        "2. Suggested coverage amount\n"
        "3. Any riders or add-ons worth considering\n\n"
        "Keep recommendations practical and tailored. Be concise — 3-4 sentences max."
    ),
)

print(f"Created: {policy_agent.name}")

Created: PolicyRecommendationAgent


## Step 2: Convert Agents to Tools

`.as_tool()` wraps the agent in a function tool interface. You can
customize the tool name and description.

In [4]:
risk_tool = risk_agent.as_tool(
    name="assess_risk",
    description="Evaluate an applicant's risk profile for life insurance underwriting",
    arg_name="applicant_profile",
    arg_description="Description of the applicant including age, health, lifestyle, and family history",
)

policy_tool = policy_agent.as_tool(
    name="recommend_policy",
    description="Recommend insurance policy types and coverage levels for an applicant",
    arg_name="applicant_needs",
    arg_description="Description of the applicant's insurance needs, budget, and risk profile",
)

print("Specialist agents converted to tools")

Specialist agents converted to tools


## Step 3: Create the Orchestrator

The underwriting agent uses the two specialist tools. It decides
when to call each one and synthesizes the final decision.

In [5]:
underwriter = make_client().as_agent(
    name="UnderwritingAgent",
    instructions=(
        "You are a senior insurance underwriter. When given an applicant's information:\n"
        "1. Use the assess_risk tool to get a risk evaluation\n"
        "2. Use the recommend_policy tool to get policy recommendations\n"
        "3. Synthesize both into a final underwriting decision\n\n"
        "Your decision should include:\n"
        "- Whether to approve, approve with conditions, or decline\n"
        "- Recommended premium tier (standard / preferred / substandard)\n"
        "- Brief rationale\n\n"
        "Be professional and structured in your response."
    ),
    tools=[risk_tool, policy_tool],
)

print(f"Orchestrator '{underwriter.name}' ready with {2} specialist tools")

Orchestrator 'UnderwritingAgent' ready with 2 specialist tools


## Demo: Full Underwriting Flow

The orchestrator will call both specialists and produce a unified decision.

In [6]:
result = await underwriter.run(
    "Evaluate this applicant for life insurance:\n"
    "- Name: David Kim\n"
    "- Age: 42\n"
    "- Gender: Male\n"
    "- Smoker: No\n"
    "- BMI: 24\n"
    "- Occupation: Software engineer (desk job)\n"
    "- Family history: Father diagnosed with Type 2 diabetes at age 55\n"
    "- Needs: Coverage for family of 4, mortgage balance of $350K\n"
    "- Budget: Around $150/month for premiums"
)

print(result)

### Underwriting Decision for David Kim

**Approval Decision**: Approved  
**Premium Tier**: Preferred  
**Rationale**:  
David Kim demonstrates a strong risk profile:
- Favorable health metrics: Normal BMI, non-smoker, and no reported chronic conditions.
- Stable and low-risk occupation (software engineer).
- Although there is some familial history of Type 2 diabetes, it manifests later in life and does not significantly impact his current risk level.  

Based on his budget and coverage needs, the recommended policy and coverage align with his goals of protecting his family financially. The term life insurance suggested with a larger coverage amount provides flexibility and comprehensive protection for both income replacement and mortgage balance coverage.




## What Just Happened?

```
User: "Evaluate this applicant..."
         │
         ▼
┌─────────────────────────────────┐
│    Underwriting Agent           │
│    (orchestrator)               │
│                                 │
│  1. Calls assess_risk tool ─────┼──► Risk Assessment Agent
│     ← receives risk rating      │    "moderate risk, family hx diabetes"
│                                 │
│  2. Calls recommend_policy ─────┼──► Policy Recommendation Agent
│     ← receives recommendation   │    "20-year term, $500K coverage"
│                                 │
│  3. Synthesizes final decision  │
│     "Approved — standard tier"  │
└─────────────────────────────────┘
```

Each specialist runs as a **separate agent invocation** — with its own
instructions and context. The orchestrator only sees the text result.

In [7]:
await credential.close()
print("Done!")

Done!


---

## Key Takeaways

- `.as_tool()` converts any agent into a function tool
- The orchestrator agent decides **when** to call each specialist
- Each specialist agent runs independently with its own instructions
- This pattern is great when you need **separation of concerns** within one conversation turn
- For more complex multi-agent choreography (e.g., review loops), see **Workflows** in notebooks 07-08

**Next up:** [06 — Middleware](./06-middleware.ipynb) — intercept agent behavior
for compliance logging, PII detection, and performance monitoring.